# V-JEPA 2 vs Qwen2.5-VL — FineGym (4 Events)

**Dataset**: FineGym — Gymnastics Event Classification
**Classes**: Floor Exercise (FX), Balance Beam (BB), Uneven Bars (UB), Vault (VT)
**Models**: V-JEPA 2 (654M) vs Qwen2.5-VL (7B)
**Method**: Frozen features + classifiers (no fine-tuning)

**Kaggle Setup**:
- Accelerator: **GPU P100** or **GPU T4 x2**
- Internet: **ON** (needed to download videos from YouTube)
- Checkpoints saved to `/kaggle/working/` every 100 clips

In [1]:
!pip install -q transformers accelerate bitsandbytes peft datasets huggingface_hub opencv-python-headless scikit-learn yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0

In [2]:
import torch, sys, os, time, json, gc, random, tempfile, subprocess, urllib.request
import numpy as np
import torch.nn as nn
from pathlib import Path
from collections import Counter
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, top_k_accuracy_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'GPU: {gpu} ({mem/1e9:.1f} GB)')
else:
    print('WARNING: No GPU!')

SEED = 42
NUM_FRAMES = 8
IMG_SIZE = 224
TRAIN_SPLIT = 0.8
CKPT_EVERY = 100
SAVE_DIR = '/kaggle/working'
CLIPS_DIR = os.path.join(SAVE_DIR, 'finegym_clips')
os.makedirs(CLIPS_DIR, exist_ok=True)

# 4 event classes (FineGym uses 1-indexed)
EVENT_NAMES = {1: 'Balance_Beam', 2: 'Uneven_Bars', 3: 'Floor_Exercise', 4: 'Vault'}
EVENT_LIST = ['Balance_Beam', 'Floor_Exercise', 'Uneven_Bars', 'Vault']  # sorted
NUM_CLASSES = 4

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f'Classes: {EVENT_LIST}')

GPU: Tesla T4 (15.6 GB)
Classes: ['Balance_Beam', 'Floor_Exercise', 'Uneven_Bars', 'Vault']


## Step 1: Download FineGym Annotations & Prepare Clips
Downloads the official annotation JSON, then uses `yt-dlp` to download YouTube clips and trim them to event timestamps.

In [3]:
# Download FineGym annotations
ANNO_URL = 'https://sdolivia.github.io/FineGym/resources/dataset/finegym_annotation_info_v1.1.json'
ANNO_PATH = os.path.join(SAVE_DIR, 'finegym_annotations.json')

if not os.path.exists(ANNO_PATH):
    print('Downloading FineGym annotations...')
    urllib.request.urlretrieve(ANNO_URL, ANNO_PATH)
    print('Done')

with open(ANNO_PATH) as f:
    annotations = json.load(f)

# Parse all event clips
all_clips = []
for yt_id, events in annotations.items():
    for event_key, info in events.items():
        event_id = info['event']  # 1=BB, 2=UB, 3=FX, 4=VT
        if event_id not in EVENT_NAMES:
            continue
        ts = info['timestamps'][0]  # [start_sec, end_sec]
        all_clips.append({
            'yt_id': yt_id,
            'event_key': event_key,
            'event_id': event_id,
            'event_name': EVENT_NAMES[event_id],
            'start': ts[0],
            'end': ts[1],
        })

print(f'Total event clips in annotations: {len(all_clips)}')
counts = Counter(c['event_name'] for c in all_clips)
for name in sorted(counts): print(f'  {name:<20} {counts[name]:>5}')

Done
Total event clips in annotations: 6915
  Balance_Beam          2436
  Floor_Exercise        1597
  Uneven_Bars           1214
  Vault                 1668


In [4]:
def download_clip(yt_id, start, end, out_path, max_retries=2):
    """Download a clip from YouTube using yt-dlp."""
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return True  # Already downloaded
    
    duration = end - start
    url = f'https://www.youtube.com/watch?v={yt_id}'
    
    for attempt in range(max_retries):
        try:
            cmd = [
                'yt-dlp', '--quiet', '--no-warnings',
                '--download-sections', f'*{start}-{end}',
                '-f', 'worst[ext=mp4]/worst',  # smallest quality (faster)
                '--force-keyframes-at-cuts',
                '-o', out_path,
                url
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if os.path.exists(out_path) and os.path.getsize(out_path) > 500:
                return True
        except (subprocess.TimeoutExpired, Exception):
            pass
    return False

def load_frames_from_file(video_path, n_frames=8, size=224):
    """Load uniformly-spaced frames from a video file."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1: cap.release(); return None
    idxs = np.linspace(0, total - 1, n_frames, dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, fr = cap.read()
        if not ok: fr = np.zeros((size, size, 3), dtype=np.uint8)
        fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        fr = cv2.resize(fr, (size, size))
        frames.append(Image.fromarray(fr))
    cap.release()
    return frames

# Download clips (with resume support)
DL_LOG = os.path.join(SAVE_DIR, 'finegym_download_log.json')
if os.path.exists(DL_LOG):
    with open(DL_LOG) as f: dl_status = json.load(f)
else:
    dl_status = {}

print(f'Downloading FineGym clips... ({len(dl_status)} already attempted)')

# Limit to manageable number — sample 300 clips per class (1200 total)
MAX_PER_CLASS = 300
sampled_clips = []
for event_name in EVENT_LIST:
    cls_clips = [c for c in all_clips if c['event_name'] == event_name]
    random.shuffle(cls_clips)
    sampled_clips.extend(cls_clips[:MAX_PER_CLASS])
random.shuffle(sampled_clips)

print(f'Sampling {len(sampled_clips)} clips ({MAX_PER_CLASS} per class)')

success, fail = 0, 0
t0 = time.time()
for i, clip in enumerate(sampled_clips):
    clip_key = f"{clip['yt_id']}_{clip['event_key']}"
    out_path = os.path.join(CLIPS_DIR, f"{clip_key}.mp4")
    
    if clip_key in dl_status:
        if dl_status[clip_key] == 'ok': success += 1
        else: fail += 1
        continue
    
    ok = download_clip(clip['yt_id'], clip['start'], clip['end'], out_path)
    dl_status[clip_key] = 'ok' if ok else 'fail'
    if ok: success += 1
    else: fail += 1
    
    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        print(f'  [{i+1}/{len(sampled_clips)}] OK: {success} | Fail: {fail} | {elapsed/60:.1f}min')
        json.dump(dl_status, open(DL_LOG, 'w'))

json.dump(dl_status, open(DL_LOG, 'w'))
print(f'\nDownload complete: {success} ok, {fail} failed')

# Build final clip list (only successful downloads)
final_clips = []
for clip in sampled_clips:
    clip_key = f"{clip['yt_id']}_{clip['event_key']}"
    out_path = os.path.join(CLIPS_DIR, f"{clip_key}.mp4")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 500:
        label_idx = EVENT_LIST.index(clip['event_name'])
        final_clips.append({'path': out_path, 'label': label_idx, 'name': clip['event_name']})

counts = Counter(c['name'] for c in final_clips)
print(f'\nReady clips: {len(final_clips)}')
for name in sorted(counts): print(f'  {name:<20} {counts[name]:>5}')

Sampling 1200 clips (300 per class)
  [20/1200] OK: 17 | Fail: 3 | 3.7min
  [40/1200] OK: 34 | Fail: 6 | 9.7min
  [60/1200] OK: 48 | Fail: 12 | 13.8min
  [80/1200] OK: 65 | Fail: 15 | 18.6min
  [100/1200] OK: 81 | Fail: 19 | 23.4min
  [120/1200] OK: 96 | Fail: 24 | 26.5min
  [140/1200] OK: 113 | Fail: 27 | 31.3min
  [160/1200] OK: 130 | Fail: 30 | 35.0min
  [180/1200] OK: 147 | Fail: 33 | 38.7min
  [200/1200] OK: 164 | Fail: 36 | 43.3min
  [220/1200] OK: 171 | Fail: 49 | 45.7min
  [240/1200] OK: 171 | Fail: 69 | 46.6min
  [260/1200] OK: 171 | Fail: 89 | 47.5min
  [280/1200] OK: 171 | Fail: 109 | 48.3min
  [300/1200] OK: 171 | Fail: 129 | 49.2min
  [320/1200] OK: 171 | Fail: 149 | 50.1min
  [340/1200] OK: 171 | Fail: 169 | 51.0min
  [360/1200] OK: 171 | Fail: 189 | 51.9min
  [380/1200] OK: 171 | Fail: 209 | 52.8min
  [400/1200] OK: 171 | Fail: 229 | 53.7min
  [420/1200] OK: 171 | Fail: 249 | 54.6min
  [440/1200] OK: 171 | Fail: 269 | 55.4min
  [460/1200] OK: 171 | Fail: 289 | 56.3min
  

## Step 2: Extract V-JEPA 2 Features
Frozen backbone → mean-pooled last hidden state (1280-dim per video). Checkpoint saved after extraction.

In [10]:
from transformers import VJEPA2Model, VJEPA2VideoProcessor

VJEPA_FEAT = os.path.join(SAVE_DIR, 'vjepa2_finegym_features.pt')
VJEPA_CKPT = os.path.join(SAVE_DIR, 'vjepa2_finegym_ckpt.pt')

if os.path.exists(VJEPA_FEAT):
    print(f'V-JEPA features COMPLETE: {VJEPA_FEAT}')
    vdata = torch.load(VJEPA_FEAT, map_location='cpu', weights_only=True)
    print(f'  {vdata["features"].shape}')
else:
    print('Loading V-JEPA 2...')
    vproc = VJEPA2VideoProcessor.from_pretrained('facebook/vjepa2-vith-fpc64-256')
    vmodel = VJEPA2Model.from_pretrained(
        'facebook/vjepa2-vith-fpc64-256',
        torch_dtype=torch.float16, attn_implementation='sdpa',
    ).to(device).eval()

    features, labels, start_idx = [], [], 0
    if os.path.exists(VJEPA_CKPT):
        ckpt = torch.load(VJEPA_CKPT, map_location='cpu', weights_only=True)
        features = list(ckpt['features'].unbind(0))
        labels = ckpt['labels'].tolist()
        start_idx = int(ckpt['next_idx'])
        print(f'  >> Resuming: {len(features)} done, starting at {start_idx}')
        del ckpt; gc.collect()

    errors = 0; t0 = time.time(); total = len(final_clips)
    for i in range(start_idx, total):
        try:
            clip = final_clips[i]
            frames = load_frames_from_file(clip['path'], NUM_FRAMES, IMG_SIZE)
            if frames is None: errors += 1; continue

            inputs = vproc(frames, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad(), torch.amp.autocast('cuda'):
                out = vmodel(**inputs)
                feat = out.last_hidden_state.mean(dim=1).cpu().float().squeeze(0)
            features.append(feat); labels.append(clip['label'])

            if (i+1) % 50 == 0:
                elapsed = time.time() - t0; done = i - start_idx + 1
                speed = done / elapsed; eta = (total - i - 1) / speed / 60
                print(f'  [{i+1}/{total}] {len(features)} done | {speed:.1f} v/s | ETA: {eta:.0f}min')

            if (i+1) % CKPT_EVERY == 0:
                torch.save({'features': torch.stack(features),
                            'labels': torch.tensor(labels, dtype=torch.long),
                            'next_idx': i+1}, VJEPA_CKPT)
                print(f'    >> Checkpoint ({len(features)} clips)')

            if (i+1) % 200 == 0: torch.cuda.empty_cache()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower(): torch.cuda.empty_cache(); errors += 1; continue
            raise
        except Exception as e:
            errors += 1
            if errors <= 3: print(f'  Error {i}: {e}')
            continue

    ft = torch.stack(features); lt = torch.tensor(labels, dtype=torch.long)
    torch.save({'features': ft, 'labels': lt}, VJEPA_FEAT)
    if os.path.exists(VJEPA_CKPT): os.remove(VJEPA_CKPT)
    print(f'V-JEPA DONE: {ft.shape} | {(time.time()-t0)/60:.1f}min | {errors} errors')

    del vmodel, vproc; torch.cuda.empty_cache(); gc.collect()

## Step 3: Extract Qwen2.5-VL Features
4-bit quantized. Checkpoint every 100 clips.

In [11]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

QWEN_FEAT = os.path.join(SAVE_DIR, 'qwen25vl_finegym_features.pt')
QWEN_CKPT = os.path.join(SAVE_DIR, 'qwen25vl_finegym_ckpt.pt')

if os.path.exists(QWEN_FEAT):
    print(f'Qwen features COMPLETE: {QWEN_FEAT}')
    qdata = torch.load(QWEN_FEAT, map_location='cpu', weights_only=True)
    print(f'  {qdata["features"].shape}')
else:
    print('Loading Qwen2.5-VL (4-bit)...')
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
    qmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2.5-VL-7B-Instruct',
        quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
    qproc = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
    qmodel.eval()
    qdev = next(qmodel.parameters()).device

    features, labels, start_idx = [], [], 0
    if os.path.exists(QWEN_CKPT):
        ckpt = torch.load(QWEN_CKPT, map_location='cpu', weights_only=True)
        features = list(ckpt['features'].unbind(0))
        labels = ckpt['labels'].tolist()
        start_idx = int(ckpt['next_idx'])
        print(f'  >> Resuming: {len(features)} done, starting at {start_idx}')
        del ckpt; gc.collect()

    errors = 0; t0 = time.time(); total = len(final_clips)
    for i in range(start_idx, total):
        try:
            clip = final_clips[i]
            frames = load_frames_from_file(clip['path'], 4, IMG_SIZE)
            if frames is None: errors += 1; continue

            messages = [{'role': 'user', 'content': [
                {'type': 'video', 'video': frames},
                {'type': 'text', 'text': 'Describe.'},
            ]}]
            text = qproc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = qproc(text=[text], videos=[frames], return_tensors='pt', padding=True)
            inputs = {k: v.to(qdev) if hasattr(v, 'to') else v for k, v in inputs.items()}

            with torch.no_grad():
                out = qmodel(**inputs, output_hidden_states=True)
                feat = out.hidden_states[-1].mean(dim=1).cpu().float().squeeze(0)
            features.append(feat); labels.append(clip['label'])

            if (i+1) % 50 == 0:
                elapsed = time.time() - t0; done = i - start_idx + 1
                speed = done / elapsed; eta = (total - i - 1) / speed / 60
                print(f'  [{i+1}/{total}] {len(features)} done | {speed:.1f} v/s | ETA: {eta:.0f}min')

            if (i+1) % CKPT_EVERY == 0:
                torch.save({'features': torch.stack(features),
                            'labels': torch.tensor(labels, dtype=torch.long),
                            'next_idx': i+1}, QWEN_CKPT)
                print(f'    >> Checkpoint ({len(features)} clips)')

            if (i+1) % 100 == 0: torch.cuda.empty_cache()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower(): torch.cuda.empty_cache(); errors += 1; continue
            raise
        except Exception as e:
            errors += 1
            if errors <= 3: print(f'  Error {i}: {e}')
            continue

    ft = torch.stack(features); lt = torch.tensor(labels, dtype=torch.long)
    torch.save({'features': ft, 'labels': lt}, QWEN_FEAT)
    if os.path.exists(QWEN_CKPT): os.remove(QWEN_CKPT)
    print(f'Qwen DONE: {ft.shape} | {(time.time()-t0)/60:.1f}min | {errors} errors')

    del qmodel, qproc; torch.cuda.empty_cache(); gc.collect()

Qwen features COMPLETE: /kaggle/working/qwen25vl_finegym_features.pt
  torch.Size([172, 3584])


## Step 4: Run Classifiers & Compare

In [7]:
vdata = torch.load(VJEPA_FEAT, map_location='cpu', weights_only=True)
qdata = torch.load(QWEN_FEAT, map_location='cpu', weights_only=True)

vf = vdata['features'].numpy(); qf = qdata['features'].numpy()
vl = vdata['labels'].numpy(); ql = qdata['labels'].numpy()
n = min(len(vl), len(ql))
vf, qf, labels = vf[:n], qf[:n], vl[:n]

print(f'V-JEPA features: {vf.shape}')
print(f'Qwen features:   {qf.shape}')
print(f'Clips: {n} | Classes: {NUM_CLASSES}')

n_tr = int(n * TRAIN_SPLIT)
perm = np.random.RandomState(SEED).permutation(n)
tr, ev = perm[:n_tr], perm[n_tr:]
y_tr, y_ev = labels[tr], labels[ev]
print(f'Train: {len(tr)} | Eval: {len(ev)}')
for i, name in enumerate(EVENT_LIST):
    print(f'  {name:<20} train={np.sum(y_tr==i):>4}  eval={np.sum(y_ev==i):>4}')

V-JEPA features: (172, 1280)
Qwen features:   (172, 3584)
Clips: 172 | Classes: 4
Train: 137 | Eval: 35
  Balance_Beam         train=  38  eval=   9
  Floor_Exercise       train=  34  eval=  11
  Uneven_Bars          train=  38  eval=   9
  Vault                train=  27  eval=   6


## Step 4b: PyTorch Neural Network Classifiers (Enhanced)
Custom neural networks for feature classification: **DeepMLP**, **ResMLP** (residual connections), **1D-CNN**, **Mini Transformer** encoder, **AttentionPoolMLP** (learned attention), and **CosineClassifier** (magnitude-invariant). All trained on PCA-reduced features with:
- **Label smoothing** (0.1) for regularization
- **Feature-space mixup** augmentation for better generalization
- Early stopping with cosine LR scheduling

In [ ]:
# ──────────────────────────────────────────────────────────────
#  PyTorch Neural Network Architectures for Feature Classification
#  Enhanced with AttentionPool, GeM pooling, and label smoothing
# ──────────────────────────────────────────────────────────────
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

LABEL_SMOOTHING = 0.1  # Regularization: prevents overconfident predictions

# ── 1. DeepMLP: 4-layer MLP with BatchNorm, Dropout, GELU ──
class DeepMLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden_dims=(512, 256, 128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


# ── 2. ResMLP: MLP with residual / skip connections ──
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))

class ResMLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=256, n_blocks=3, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 3. MiniTransformer: Transformer encoder on chunked features ──
class MiniTransformer(nn.Module):
    def __init__(self, in_dim, num_classes, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.chunk_size = d_model
        # Pad feature dim to be divisible by d_model
        self.n_chunks = (in_dim + d_model - 1) // d_model
        self.pad_dim = self.n_chunks * d_model
        self.proj = nn.Linear(d_model, d_model)  # per-chunk linear
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_chunks + 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.size(0)
        # Pad and reshape into chunks: (B, n_chunks, d_model)
        if x.size(1) < self.pad_dim:
            x = F.pad(x, (0, self.pad_dim - x.size(1)))
        x = x.view(B, self.n_chunks, self.chunk_size)
        x = self.proj(x)
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, n_chunks+1, d_model)
        x = x + self.pos_embed
        x = self.encoder(x)
        x = self.norm(x[:, 0])  # CLS token output
        return self.head(x)


# ── 4. AttentionPoolMLP: MLP with learned attention pooling ──
# Learns which feature dimensions are most discriminative
class AttentionPoolMLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=256, dropout=0.3):
        super().__init__()
        # Split features into chunks and apply attention across them
        self.chunk_size = 64
        self.n_chunks = (in_dim + self.chunk_size - 1) // self.chunk_size
        self.pad_dim = self.n_chunks * self.chunk_size
        # Attention over chunks
        self.attn_proj = nn.Linear(self.chunk_size, 1)
        self.backbone = nn.Sequential(
            nn.Linear(self.chunk_size, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):
        B = x.size(0)
        if x.size(1) < self.pad_dim:
            x = F.pad(x, (0, self.pad_dim - x.size(1)))
        x = x.view(B, self.n_chunks, self.chunk_size)  # (B, C, chunk)
        # Attention-weighted pooling across chunks
        attn_w = torch.softmax(self.attn_proj(x), dim=1)  # (B, C, 1)
        x = (x * attn_w).sum(dim=1)  # (B, chunk_size)
        return self.head(self.backbone(x))


# ── 5. CosineClassifier: Cosine-similarity based head ──
# Removes magnitude bias — only direction matters (great for normalized features)
class CosineClassifier(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=256, scale=16.0, dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.weight = nn.Parameter(torch.randn(num_classes, hidden))
        self.scale = nn.Parameter(torch.tensor(scale))

    def forward(self, x):
        x = self.backbone(x)
        # Cosine similarity: normalize both feature and weight vectors
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        return self.scale * x_norm @ w_norm.T


# ── Training harness with label smoothing + mixup ──
def mixup_data(x, y, alpha=0.2):
    """Feature-space mixup: interpolate training samples for better generalization."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # Ensure lam >= 0.5
    idx = torch.randperm(x.size(0))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def train_pytorch_model(model, X_tr, y_tr, X_ev, y_ev,
                        epochs=200, lr=1e-3, batch_size=32, patience=20, 
                        use_mixup=True, verbose=False):
    """Train a PyTorch classifier with early stopping, label smoothing, and mixup."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    # Convert to tensors
    Xt = torch.tensor(X_tr, dtype=torch.float32)
    yt = torch.tensor(y_tr, dtype=torch.long)
    Xe = torch.tensor(X_ev, dtype=torch.float32).to(device)
    ye = torch.tensor(y_ev, dtype=torch.long).to(device)

    ds = TensorDataset(Xt, yt)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    best_acc, best_state, wait = 0, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            if use_mixup and xb.size(0) > 1:
                xb_mix, ya, yb_mix, lam = mixup_data(xb, yb, alpha=0.2)
                logits = model(xb_mix)
                loss = lam * F.cross_entropy(logits, ya, label_smoothing=LABEL_SMOOTHING) + \
                       (1 - lam) * F.cross_entropy(logits, yb_mix, label_smoothing=LABEL_SMOOTHING)
            else:
                loss = F.cross_entropy(model(xb), yb, label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()

        # Eval
        model.eval()
        with torch.no_grad():
            logits = model(Xe)
            acc = (logits.argmax(1) == ye).float().mean().item()
        if acc > best_acc:
            best_acc = acc; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose: print(f'    Early stop at epoch {epoch+1}')
                break

    # Load best and predict
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    model = model.to(device)
    with torch.no_grad():
        logits = model(Xe)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
    return preds, probs

print('PyTorch neural network architectures defined ✓')
print('  • DeepMLP         — 4-layer MLP with BatchNorm + GELU + Dropout')
print('  • ResMLP          — MLP with residual skip connections (3 blocks)')
print('  • MiniTransformer — Transformer encoder on chunked features')
print('  • AttentionPoolMLP— MLP with learned attention pooling')
print('  • CosineClassifier— Cosine-similarity head (magnitude-invariant)')
print(f'  • Label smoothing = {LABEL_SMOOTHING}')
print(f'  • Feature-space mixup augmentation enabled')

In [8]:
def run_classifiers(X_tr, y_tr, X_ev, y_ev):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_ev_s = scaler.transform(X_ev)
    pca = PCA(n_components=0.95, random_state=SEED)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_ev_p = pca.transform(X_ev_s)
    in_dim = X_tr_p.shape[1]
    print(f'  PCA: {X_tr_s.shape[1]} -> {in_dim} dims')

    # ── Sklearn classifiers ──
    classifiers = {
        'SVM-RBF':      SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced',
                            random_state=SEED, probability=True),
        'KNN-7':        KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
        'RandomForest': RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                               random_state=SEED, n_jobs=-1),
        'MLP':          MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=1000, alpha=1e-4,
                                      random_state=SEED, early_stopping=True, learning_rate='adaptive'),
        'LogReg':       LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000,
                                           random_state=SEED, multi_class='multinomial'),
    }

    print(f'\n  {"Clf":<20} {"Acc":>7} {"F1":>7} {"MCC":>7}  {"Type"}')
    print(f'  {"-"*55}')

    best_name, best_f1, best_preds, best_prob = '', 0, None, None

    # Run sklearn classifiers
    for cname, clf in classifiers.items():
        print(f'  Training {cname}...', end=' ', flush=True)
        clf.fit(X_tr_p, y_tr)
        preds = clf.predict(X_ev_p)
        prob = clf.predict_proba(X_ev_p) if hasattr(clf, 'predict_proba') else None
        acc = accuracy_score(y_ev, preds)
        f1 = f1_score(y_ev, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(y_ev, preds)
        mark = ' *' if f1 > best_f1 else ''
        if f1 > best_f1:
            best_f1 = f1; best_name = cname; best_preds = preds; best_prob = prob
        print(f'\r  {cname:<20} {acc*100:6.1f}% {f1*100:6.1f}% {mcc:6.3f}{mark}  sklearn')

    # ── PyTorch neural network classifiers ──
    torch.manual_seed(SEED)
    nn_models = {
        'DeepMLP':        DeepMLP(in_dim, NUM_CLASSES, hidden_dims=(512, 256, 128, 64), dropout=0.3),
        'ResMLP':         ResMLP(in_dim, NUM_CLASSES, hidden=256, n_blocks=3, dropout=0.2),
        'Transformer':    MiniTransformer(in_dim, NUM_CLASSES, d_model=64, nhead=4, num_layers=2, dropout=0.2),
        'AttnPoolMLP':    AttentionPoolMLP(in_dim, NUM_CLASSES, hidden=256, dropout=0.3),
        'CosineClf':      CosineClassifier(in_dim, NUM_CLASSES, hidden=256, scale=16.0, dropout=0.3),
    }

    for cname, model in nn_models.items():
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  Training {cname} ({n_params:,} params)...', end=' ', flush=True)
        torch.manual_seed(SEED)
        preds, prob = train_pytorch_model(
            model, X_tr_p, y_tr, X_ev_p, y_ev,
            epochs=300, lr=1e-3, batch_size=32, patience=30,
        )
        acc = accuracy_score(y_ev, preds)
        f1 = f1_score(y_ev, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(y_ev, preds)
        mark = ' *' if f1 > best_f1 else ''
        if f1 > best_f1:
            best_f1 = f1; best_name = cname; best_preds = preds; best_prob = prob
        print(f'\r  {cname:<20} {acc*100:6.1f}% {f1*100:6.1f}% {mcc:6.3f}{mark}  pytorch')

    # ── Compute full metrics for best ──
    m = dict(
        accuracy=accuracy_score(y_ev, best_preds),
        precision_macro=precision_score(y_ev, best_preds, average='macro', zero_division=0),
        precision_weighted=precision_score(y_ev, best_preds, average='weighted', zero_division=0),
        recall_macro=recall_score(y_ev, best_preds, average='macro', zero_division=0),
        recall_weighted=recall_score(y_ev, best_preds, average='weighted', zero_division=0),
        f1_macro=f1_score(y_ev, best_preds, average='macro', zero_division=0),
        f1_weighted=f1_score(y_ev, best_preds, average='weighted', zero_division=0),
        mcc=matthews_corrcoef(y_ev, best_preds),
        cohen_kappa=cohen_kappa_score(y_ev, best_preds),
    )
    report = classification_report(y_ev, best_preds, target_names=EVENT_LIST, zero_division=0)
    cm = confusion_matrix(y_ev, best_preds)
    return best_name, m, report, cm

print('=' * 60)
print('  V-JEPA 2 (654M) \u2014 Self-supervised video encoder')
print('=' * 60)
vj_name, vj_m, vj_report, vj_cm = run_classifiers(vf[tr], y_tr, vf[ev], y_ev)
print(f'\n  Best: {vj_name}')
print(vj_report)

print('=' * 60)
print('  Qwen2.5-VL (7B) \u2014 Vision-Language Model')
print('=' * 60)
qw_name, qw_m, qw_report, qw_cm = run_classifiers(qf[tr], y_tr, qf[ev], y_ev)
print(f'\n  Best: {qw_name}')
print(qw_report)

## Step 5: Final Comparison — FineGym (4 Events)

In [9]:
print('=' * 62)
print('  FINAL COMPARISON \u2014 FineGym (4 Event Classes)')
print('  Balance Beam | Floor Exercise | Uneven Bars | Vault')
print('=' * 62)

print(f'\n  {"Metric":<25} {"V-JEPA 2 (654M)":>18} {"Qwen2.5-VL (7B)":>18}')
print(f'  {"-"*61}')

def row(name, v1, v2, pct=True):
    if pct: print(f'  {name:<25} {v1*100:>17.1f}% {v2*100:>17.1f}%')
    else:   print(f'  {name:<25} {v1:>18.3f} {v2:>18.3f}')

row('Accuracy',            vj_m['accuracy'],           qw_m['accuracy'])
row('Precision (macro)',   vj_m['precision_macro'],    qw_m['precision_macro'])
row('Precision (weighted)',vj_m['precision_weighted'],  qw_m['precision_weighted'])
row('Recall (macro)',      vj_m['recall_macro'],       qw_m['recall_macro'])
row('Recall (weighted)',   vj_m['recall_weighted'],    qw_m['recall_weighted'])
row('F1 (macro)',          vj_m['f1_macro'],           qw_m['f1_macro'])
row('F1 (weighted)',       vj_m['f1_weighted'],        qw_m['f1_weighted'])
row('MCC',                 vj_m['mcc'],                qw_m['mcc'], pct=False)
row("Cohen's Kappa",       vj_m['cohen_kappa'],        qw_m['cohen_kappa'], pct=False)

print(f'\n  {"-"*61}')
print(f'  {"Architecture":<25} {"V-JEPA 2":>18} {"Qwen2.5-VL":>18}')
print(f'  {"-"*61}')
print(f'  {"Total params":<25} {"654M":>18} {"7B":>18}')
print(f'  {"Feature dim":<25} {str(vf.shape[1]):>18} {str(qf.shape[1]):>18}')
print(f'  {"Best classifier":<25} {vj_name:>18} {qw_name:>18}')
print(f'  {"Paradigm":<25} {"Self-supervised":>18} {"Autoregressive":>18}')
print(f'  {"Modality":<25} {"Video only":>18} {"Vision+Language":>18}')

vj_f1 = vj_m['f1_macro']; qw_f1 = qw_m['f1_macro']
winner = 'V-JEPA 2' if vj_f1 > qw_f1 else 'Qwen2.5-VL'
margin = abs(vj_f1 - qw_f1) * 100

print(f'\n{"="*62}')
print(f'  WINNER: {winner} (by {margin:.1f}% F1)')
print(f'{"="*62}')

# Confusion matrices
for name, cm in [('V-JEPA 2', vj_cm), ('Qwen2.5-VL', qw_cm)]:
    print(f'\n  {name} Confusion Matrix:')
    print(f'  {"":>18}', end='')
    for c in EVENT_LIST: print(f' {c[:8]:>8}', end='')
    print()
    for i, c in enumerate(EVENT_LIST):
        print(f'  {c:<18}', end='')
        for j in range(NUM_CLASSES): print(f' {cm[i][j]:>8}', end='')
        print()

# Save results
results = {
    'dataset': 'FineGym', 'task': '4-event classification',
    'classes': EVENT_LIST, 'num_clips': n,
    'train': len(tr), 'eval': len(ev),
    'vjepa2': {'params': '654M', 'best_clf': vj_name,
               **{k: float(v) for k, v in vj_m.items()}},
    'qwen25vl': {'params': '7B', 'best_clf': qw_name,
                 **{k: float(v) for k, v in qw_m.items()}},
    'winner': winner,
}
results_path = os.path.join(SAVE_DIR, 'finegym_comparison.json')
json.dump(results, open(results_path, 'w'), indent=2)
print(f'\nResults saved: {results_path}')

  FINAL COMPARISON — FineGym (4 Event Classes)
  Balance Beam | Floor Exercise | Uneven Bars | Vault

  Metric                       V-JEPA 2 (654M)    Qwen2.5-VL (7B)
  -------------------------------------------------------------
  Accuracy                               97.1%              80.0%
  Precision (macro)                      97.5%              80.6%
  Precision (weighted)                   97.4%              80.5%
  Recall (macro)                         95.8%              79.7%
  Recall (weighted)                      97.1%              80.0%
  F1 (macro)                             96.4%              79.7%
  F1 (weighted)                          97.1%              79.8%
  MCC                                    0.962              0.731
  Cohen's Kappa                          0.961              0.729

  -------------------------------------------------------------
  Architecture                        V-JEPA 2         Qwen2.5-VL
  -----------------------------------------